In [1]:
from qiskit import QuantumCircuit
import qiskit
from qiskit import transpile
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Operator
import math
from math import pi

from sqlalchemy.orm import joinedload
from sqlalchemy import select, func

from benchmarklib import BenchmarkDatabase
from benchmarklib.runners import BatchQueue

from experiments import ExperimentProblem, ExperimentTrial

TAG = "quest_huge_default"

In [2]:
service = QiskitRuntimeService()
backend = service.backend("ibm_rensselaer")

db = BenchmarkDatabase("experiments.db", ExperimentProblem, ExperimentTrial)

In [3]:
db.query(select(func.count(ExperimentProblem.id)).where(ExperimentProblem.tag == TAG))

[7000]

In [3]:
import pickle
quest_raw_filepath = "/home/eriku/projects/torchquantum/examples/quest/data/raw_data_qasm/huge.data"
with open(quest_raw_filepath, "rb") as f:
    quest_raw_data = pickle.load(f)

In [32]:
def create_trial(circuit: QuantumCircuit, idx: int) -> ExperimentTrial:
    # transform circuit according to torchquantum/examples/quest/utils/gen_app_data_real.py
    # so that fidelity is measured as the number of all-zero state measurement
    circ_app = circuit.copy()  # copy and rename to match naming of original script
    circ_app.remove_final_measurements()
    #transpiled = transpile(circ_app, backend=backend)
    circ_app.barrier()
    appended = circ_app.compose(circ_app.inverse())
    appended.cregs.clear()  # Needed to add this in newer qiskit to clear classical registers before applying measurement
    appended.measure_active()  # Note: this ends up measuring all qubits because of the barrier, but this matches the code from QuEst so we'll go with it
    appended = transpile(appended, backend=backend)
    n = appended.num_clbits
    
    experiment = ExperimentProblem(
        name=f"huge.data[{idx}]",
        tag=TAG,
        n=n
    )
    # Note: circuit_pretranspile is the original circuit before appending inverse and transpiling
    trial = ExperimentTrial(problem=experiment, circuit=appended, circuit_pretranspile=circuit, extra_data={"idx": idx, "backend": backend.name})
    return trial

In [33]:
with BatchQueue(db, backend=backend, shots=4096) as q:
    for idx in range(len(quest_raw_data)):
        circuit = qiskit.QuantumCircuit.from_qasm_str(quest_raw_data[idx][0])
        run_simulation = True if circuit.num_qubits <= 6 else False
        trial = create_trial(circuit, idx)
        q.enqueue(trial, trial.circuit, run_simulation=run_simulation)

In [ ]:
raise Exception("This block clears counts")
from sqlalchemy import update
ids = db.query(
    select(ExperimentTrial.id)
    .join(ExperimentTrial.problem)
    .where(ExperimentProblem.tag == TAG)
)
with db.session() as session:
    statement = update(ExperimentTrial).where(ExperimentTrial.id.in_(ids)).values(counts=None)
    session.execute(statement)
    session.commit()

In [6]:
await db.update_all_pending_results(service=service, result_register="meas")

In [ ]:
# manual update (since at first we messed up which result register to use)
from sqlalchemy import update
job_ids = db.query(select(ExperimentTrial.job_id).join(ExperimentTrial.problem).where(ExperimentProblem.tag == TAG).distinct())
for job_id in job_ids:
    retrieved = service.job(job_id)
    results = retrieved.result()

    trials_to_update = db.query(
        select(ExperimentTrial)
        .where(ExperimentTrial.job_id == job_id)
    )
    updates = []
    for trial in trials_to_update:
        pub_result = results[trial.job_pub_idx]
        counts = pub_result.data.meas.get_counts()
        updates.append({"id": trial.id, "counts": counts})

    with db.session() as session:
        session.execute(
            update(ExperimentTrial), updates
        )
        session.commit()

### Convert to QuEst Input Form

In [118]:
import os
import shutil

def calculate_fidelity(trial):
    # Note: this checks all the measured qubits, which is not just the ones that have gates applied (since measure_active applied measurements to qubits that had a barrier on them)
    n = len(trial.counts.keys().__iter__().__next__())
    counts = trial.counts
    if counts is None:
        return 0.0
    all_zero_key = "0" * n
    all_zero_counts = counts.get(all_zero_key, 0)
    fidelity = all_zero_counts / sum(counts.values())
    return fidelity

def calculate_fidelity2(trial):
    # this version only checks the qubits that had gates applied to them
    # first determine which qubits are actually used in the circuit
    # and identify their mapping to classical bits that will show up in the measurements
    qc = trial.circuit
    active_qubits = set()
    qubit_mapping = {}
    for instruction in qc.data:
        qargs = instruction.qubits
        cargs = instruction.clbits
        if instruction.name == "measure":
            qubit_mapping[qc.find_bit(qargs[0]).index] = qc.find_bit(cargs[0]).index
        if instruction.name in ['measure', 'barrier']:
            continue
        for qubit in instruction.qubits:
            active_qubits.add(qc.find_bit(qubit).index)

    # We found that each of the circuits contain a c register with size 1 that does not actually measure anything
    # but it does change the clbit index mapping
    # so we need to verify that this trial matches this assumption
    if qc.cregs[0].name != "c":
        raise Exception(f"Trial {trial.id} has different classical register structure")

    # build a mask to only check for zeros on the active qubits
    mask = 0
    for q in active_qubits:
        # classical bits are 0-indexed, but since the c register bit is not used for measurement, the smallest meas bit index is 1, and we need to shift everything by 1 to the right
        mask |= (1 << (qubit_mapping[q] - 1))  

    num_correct = 0
    for measurement, count in trial.counts.items():
        # measurement is already in bit order where classical bit 0 is on the right
        if (int(measurement, 2) & mask) == 0:
            num_correct += count

    fidelity = num_correct / sum(trial.counts.values())
    return fidelity

GATE_DICT = {"rz": 0, "x": 1, "sx": 2, "cx": 3, "ecr": 3}  # add ecr for noise reporting
def build_my_noise_dict(prop):  # from torchquantum.examples.quest.utils.circ_dag_converter.py
    mydict = {}
    mydict["qubit"] = {}
    mydict["gate"] = {}
    for i, qubit_prop in enumerate(prop["qubits"]):
        mydict["qubit"][i] = {}
        for item in qubit_prop:
            if item["name"] == "T1":
                mydict["qubit"][i]["T1"] = item["value"]
            elif item["name"] == "T2":
                mydict["qubit"][i]["T2"] = item["value"]
            elif item["name"] == "prob_meas0_prep1":
                mydict["qubit"][i]["prob_meas0_prep1"] = item["value"]
            elif item["name"] == "prob_meas1_prep0":
                mydict["qubit"][i]["prob_meas1_prep0"] = item["value"]
    for gate_prop in prop["gates"]:
        if not gate_prop["gate"] in GATE_DICT:
            continue
        qubit_list = tuple(gate_prop["qubits"])
        if qubit_list not in mydict["gate"]:
            mydict["gate"][qubit_list] = {}
        for item in gate_prop["parameters"]:
            if item["name"] == "gate_error":
                mydict["gate"][qubit_list][gate_prop["gate"]] = item["value"]
    return mydict

properties_dict = backend.properties().to_dict()
noise_properties = build_my_noise_dict(properties_dict)
def convert(trial):
    circuit = trial.circuit
    fidelity = calculate_fidelity2(trial)
    circuit.remove_final_measurements(inplace=True)  # model does not support the measure gate
    qasm_str = qiskit.qasm2.dumps(circuit)
    return (qasm_str, noise_properties, fidelity)

In [119]:

import pickle

dataset_filename = "quest_rerun.data"
BATCH_SIZE = 500

# first collect the random sample of trial ids to use
ids = db.query(
    select(ExperimentTrial.id)
    .join(ExperimentTrial.problem)
    .where(ExperimentProblem.tag == TAG)
)
num_trials = len(ids)

# load and convert data in batches and store temporarily on disk
if not os.path.exists(".temp_batches") and not os.path.isdir(".temp_batches"):
    os.mkdir(".temp_batches")

for batch_idx in range(0, num_trials, BATCH_SIZE):
    batch_ids = ids[batch_idx : batch_idx + BATCH_SIZE]
    trials = db.query(
        select(ExperimentTrial)
        .where(ExperimentTrial.id.in_(batch_ids))
        .options(joinedload(ExperimentTrial.problem))
    )
    print(f"Processing batch {batch_idx // BATCH_SIZE + 1} / {(num_trials + BATCH_SIZE - 1) // BATCH_SIZE}")
    raw_batch = list(map(convert, trials))
    with open(os.path.join(".temp_batches", f"batch_{batch_idx // BATCH_SIZE}.data"), "wb") as f:
        pickle.dump(raw_batch, f)

# now combine all batches into the final dataset file
raw = []
for filename in os.listdir(".temp_batches"):
    with open(os.path.join(".temp_batches", filename), "rb") as f:
        raw_batch = pickle.load(f)
        raw.extend(raw_batch)

with open(dataset_filename, "wb") as f:
    pickle.dump(raw, f)

shutil.rmtree(".temp_batches")

Processing batch 1 / 14
Processing batch 2 / 14
Processing batch 3 / 14
Processing batch 4 / 14
Processing batch 5 / 14
Processing batch 6 / 14
Processing batch 7 / 14
Processing batch 8 / 14
Processing batch 9 / 14
Processing batch 10 / 14
Processing batch 11 / 14
Processing batch 12 / 14
Processing batch 13 / 14
Processing batch 14 / 14


In [115]:
trial = db.query(select(ExperimentTrial).where(ExperimentTrial.id == 1661))[0]
trial.job_pub_idx

0